# Unidad 3: Búsqueda, Optimización y Agentes Inteligentes
## 📋 Planificación con PDDL — pyperplan
### Inteligencia Artificial — Lic. en Sistemas — FCAD/UNER

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/ia-ls-fcad-uner/blob/main/notebooks/ml/search/05_Planificacion_PDDL.ipynb)

---

### 🎯 Objetivos
- Entender la diferencia entre **búsqueda** y **planificación** en IA.
- Aprender la sintaxis de **PDDL** (Planning Domain Definition Language).
- Modelar el dominio clásico **Mundo de Bloques** (Blocksworld).
- Resolver instancias del problema con **BFS** y **A*** usando `pyperplan`.
- Comparar planificadores según longitud del plan y eficiencia.

---

### 📖 Teoría

**Planificación automática** es la rama de IA que se ocupa de generar secuencias de acciones (**planes**) para alcanzar un objetivo dado desde un estado inicial.

A diferencia de la búsqueda heurística, la planificación trabaja con **representaciones simbólicas** del dominio.

**PDDL** es el lenguaje estándar para definir problemas de planificación. Tiene dos componentes:

| Componente | Contenido |
|------------|----------|
| **Dominio** | Predicados, acciones (precondiciones + efectos) |
| **Problema** | Objetos, estado inicial, objetivo |

Cada acción tiene la forma:
```
(:action nombre
  :parameters (?x ?y ...)
  :precondition (and ...)
  :effect       (and (not ...) ...)
)
```

**pyperplan** es un planificador PDDL en Python que implementa:
- Búsqueda en amplitud (BFS)
- A* con múltiples heurísticas (hFF, hMax, hAdd, landmarks)
- Búsqueda en profundidad iterativa (GBFS)

---

### 🏗️ Dominio: Mundo de Bloques (Blocksworld)

Un brazo robótico puede agarrar y apilar bloques sobre una mesa.

```
Estado inicial:    Estado objetivo:

  [C]                 [A]
  [B]                 [B]
  [A]                 [C]
 _____               _____
 MESA                MESA
```

**Predicados**: `on(x,y)`, `ontable(x)`, `clear(x)`, `holding(x)`, `handempty`  
**Acciones**: `pick-up`, `put-down`, `stack`, `unstack`

## 📦 Paso 1: Instalación de pyperplan

In [ ]:
# Instalar pyperplan si no está disponible
try:
    import pyperplan
    print(f"✅ pyperplan ya instalado: {pyperplan.__version__}")
except (ImportError, AttributeError):
    import subprocess
    subprocess.run(['pip', 'install', 'pyperplan', '-q'], check=True)
    print("✅ pyperplan instalado")

In [ ]:
import subprocess
import tempfile
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display

print("✅ Librerías importadas")

## 📝 Paso 2: Sintaxis PDDL explicada

A continuación se muestra el **dominio Blocksworld** con comentarios en cada sección.

In [ ]:
# ============================================================
# Dominio PDDL: Blocksworld
# ============================================================
DOMAIN_PDDL = """
(define (domain blocksworld)

  ;; Requerimientos: lógica proposicional básica (STRIPS)
  (:requirements :strips)

  ;; Predicados del dominio:
  ;;   on(x, y)    -> bloque x está sobre bloque y
  ;;   ontable(x)  -> bloque x está sobre la mesa
  ;;   clear(x)    -> la parte superior de x está libre
  ;;   holding(x)  -> el brazo sostiene el bloque x
  ;;   handempty   -> el brazo está vacío
  (:predicates
    (on ?x ?y)
    (ontable ?x)
    (clear ?x)
    (holding ?x)
    (handempty)
  )

  ;; pick-up: levantar un bloque de la mesa
  ;;   Pre: bloque x está sobre la mesa, su tope es libre, brazo vacío
  ;;   Efecto: x ya no está en la mesa, el brazo lo sostiene
  (:action pick-up
    :parameters (?x)
    :precondition (and (clear ?x) (ontable ?x) (handempty))
    :effect (and (not (ontable ?x)) (not (clear ?x)) (not (handempty)) (holding ?x))
  )

  ;; put-down: dejar un bloque en la mesa
  (:action put-down
    :parameters (?x)
    :precondition (holding ?x)
    :effect (and (not (holding ?x)) (clear ?x) (ontable ?x) (handempty))
  )

  ;; stack: apilar bloque x sobre bloque y
  ;;   Pre: brazo sostiene x, y tiene el tope libre
  (:action stack
    :parameters (?x ?y)
    :precondition (and (holding ?x) (clear ?y))
    :effect (and (not (holding ?x)) (not (clear ?y)) (clear ?x) (handempty) (on ?x ?y))
  )

  ;; unstack: quitar bloque x que está sobre y
  ;;   Pre: x está sobre y, el tope de x está libre, brazo vacío
  (:action unstack
    :parameters (?x ?y)
    :precondition (and (on ?x ?y) (clear ?x) (handempty))
    :effect (and (holding ?x) (clear ?y) (not (clear ?x)) (not (handempty)) (not (on ?x ?y)))
  )
)
"""

print("Dominio PDDL definido:")
print(DOMAIN_PDDL)

## 🧩 Paso 3: Definición de problemas

Se definen tres instancias con creciente complejidad.

In [ ]:
# ============================================================
# Problema 1 (fácil): apilar A sobre B
# Inicial: A en la mesa, B en la mesa
# Objetivo: A sobre B
# ============================================================
PROBLEM_1 = """
(define (problem apilar-a-sobre-b)
  (:domain blocksworld)
  (:objects a b)
  (:init
    (ontable a) (ontable b)
    (clear a)   (clear b)
    (handempty)
  )
  (:goal (and (on a b)))
)
"""

# ============================================================
# Problema 2 (medio): invertir pila de 3
# Inicial: C sobre B, B sobre A, A en la mesa  →  pila [A,B,C]
# Objetivo: A sobre B, B sobre C              →  pila [C,B,A]
# ============================================================
PROBLEM_2 = """
(define (problem invertir-pila)
  (:domain blocksworld)
  (:objects a b c)
  (:init
    (ontable a) (on b a) (on c b)
    (clear c)
    (handempty)
  )
  (:goal (and (on a b) (on b c)))
)
"""

# ============================================================
# Problema 3 (complejo): reorganizar 4 bloques
# Inicial: D-C apilados, A-B apilados (ambas pilas en la mesa)
# Objetivo: A sobre B sobre C sobre D (una sola torre)
# ============================================================
PROBLEM_3 = """
(define (problem torre-4)
  (:domain blocksworld)
  (:objects a b c d)
  (:init
    (ontable d) (on c d)
    (ontable b) (on a b)
    (clear c)   (clear a)
    (handempty)
  )
  (:goal (and (on a b) (on b c) (on c d) (ontable d)))
)
"""

PROBLEMAS = {
    'Problema 1 — Apilar A sobre B (2 bloques)':       PROBLEM_1,
    'Problema 2 — Invertir pila de 3':                 PROBLEM_2,
    'Problema 3 — Torre de 4 bloques':                 PROBLEM_3,
}

print("✅ Tres problemas PDDL definidos")
for nombre in PROBLEMAS:
    print(f"   · {nombre}")

## 🔧 Paso 4: Función auxiliar de planificación

`pyperplan` se invoca como herramienta de línea de comandos. La función `planificar()` escribe los archivos PDDL a disco, ejecuta pyperplan y parsea el resultado.

In [ ]:
def planificar(domain_pddl, problem_pddl, search='bfs', heuristic=None, verbose=True):
    """
    Resuelve un problema PDDL con pyperplan.

    Args:
        domain_pddl : string con el dominio PDDL
        problem_pddl: string con el problema PDDL
        search      : algoritmo de búsqueda ('bfs', 'astar', 'gbfs', 'wastar')
        heuristic   : heurística para A* ('hff', 'hmax', 'hadd', 'hsa', 'lmcut')
        verbose     : imprimir resultados

    Returns:
        dict con 'plan', 'length', 'time'
    """
    # Escribir archivos temporales
    tmpdir = tempfile.mkdtemp()
    domain_file  = os.path.join(tmpdir, 'domain.pddl')
    problem_file = os.path.join(tmpdir, 'problem.pddl')
    soln_file    = problem_file + '.soln'

    with open(domain_file,  'w') as f:
        f.write(domain_pddl)
    with open(problem_file, 'w') as f:
        f.write(problem_pddl)

    # Armar el comando
    cmd = ['pyperplan']
    if search:
        cmd += ['-s', search]
    if heuristic:
        cmd += ['-H', heuristic]
    cmd += [domain_file, problem_file]

    # Ejecutar
    t0 = time.perf_counter()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.perf_counter() - t0

    # Leer el plan del archivo .soln
    plan = []
    if os.path.exists(soln_file):
        with open(soln_file) as f:
            plan = [line.strip() for line in f if line.strip() and not line.startswith(';')]

    if verbose:
        busq_label = search.upper()
        if heuristic:
            busq_label += f' + {heuristic.upper()}'
        print(f"\n{'='*55}")
        print(f"  Algoritmo : {busq_label}")
        print(f"  Tiempo    : {elapsed*1000:.1f} ms")
        if plan:
            print(f"  Pasos del plan ({len(plan)} acciones):")
            for i, accion in enumerate(plan, 1):
                print(f"    Paso {i:2}: {accion}")
        else:
            print("  ⚠️  No se encontró solución (revisa stderr):")
            print(proc.stderr[-400:] if proc.stderr else proc.stdout[-400:])

    # Limpiar
    for f in [domain_file, problem_file, soln_file]:
        try:
            os.remove(f)
        except FileNotFoundError:
            pass
    try:
        os.rmdir(tmpdir)
    except OSError:
        pass

    return {'plan': plan, 'length': len(plan), 'time_ms': elapsed * 1000}


print("✅ Función planificar() lista")

## 🔍 Paso 5: Resolver Problema 1 — BFS

In [ ]:
print("Problema 1: Apilar A sobre B")
print("  Estado inicial: A en la mesa, B en la mesa")
print("  Objetivo: A sobre B")
print()

res_p1_bfs = planificar(DOMAIN_PDDL, PROBLEM_1, search='bfs')

## 🔄 Paso 6: Visualización del plan — Problema 1

Mostramos cómo evoluciona el estado a medida que se ejecuta cada acción.

In [ ]:
def visualizar_blocksworld(estados, titulos, figsize=(14, 3)):
    """
    Visualiza una secuencia de estados del mundo de bloques.
    estados: lista de listas de pilas, donde cada pila es de abajo a arriba.
             Ejemplo: [['a', 'b'], ['c']] = dos pilas, primera tiene b sobre a.
    """
    n = len(estados)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]

    colores = {'a': '#3498db', 'b': '#e74c3c', 'c': '#2ecc71', 'd': '#f39c12',
               'e': '#9b59b6', 'brazo': '#95a5a6'}

    for ax, pilas, titulo in zip(axes, estados, titulos):
        ax.set_xlim(0, 3)
        ax.set_ylim(0, 5)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(titulo, fontsize=9, fontweight='bold')

        # Mesa
        ax.add_patch(patches.Rectangle((0, 0), 3, 0.15, color='#7f8c8d'))
        ax.text(1.5, -0.25, 'MESA', ha='center', va='top', fontsize=8, color='gray')

        # Pilas de bloques
        for col_idx, pila in enumerate(pilas):
            x_base = 0.1 + col_idx * 1.0
            for row_idx, bloque in enumerate(pila):
                if bloque is None:
                    continue
                y = 0.15 + row_idx * 0.9
                color = colores.get(bloque.lower(), '#bdc3c7')
                rect = patches.FancyBboxPatch(
                    (x_base, y), 0.8, 0.8,
                    boxstyle='round,pad=0.05',
                    facecolor=color, edgecolor='black', linewidth=1.5
                )
                ax.add_patch(rect)
                ax.text(x_base + 0.4, y + 0.4, bloque.upper(),
                        ha='center', va='center', fontsize=14, fontweight='bold', color='white')

    plt.tight_layout()
    plt.show()


# Problema 1: A sobre B
# Plan: (pick-up a), (stack a b)
estados_p1 = [
    [['a'], ['b']],          # Estado inicial: A y B en la mesa (pilas separadas)
    [[], ['b']],             # Después de pick-up a: brazo sostiene A, B en la mesa
    [['b', 'a'], []],        # Después de stack a b: A sobre B
]
titulos_p1 = [
    'Estado inicial',
    'Paso 1: pick-up a',
    'Paso 2: stack a b  ✅',
]
visualizar_blocksworld(estados_p1, titulos_p1)

## ⭐ Paso 7: Resolver Problema 2 — BFS y A* + hFF

Invertir una pila de 3 bloques requiere deshacerla completamente antes de reconstruirla.

In [ ]:
print("Problema 2: Invertir pila [A, B, C] → [C, B, A]")
print("  Estado inicial: A en la mesa, B sobre A, C sobre B  (de abajo a arriba: A-B-C)")
print("  Objetivo: A sobre B, B sobre C  (de abajo a arriba: C-B-A)")
print()

res_p2_bfs  = planificar(DOMAIN_PDDL, PROBLEM_2, search='bfs')
res_p2_astar = planificar(DOMAIN_PDDL, PROBLEM_2, search='astar', heuristic='hff')

In [ ]:
# Visualización del plan del Problema 2
# Plan óptimo (6 pasos):
#   unstack c b → put-down c → unstack b a → put-down b → pick-up a → stack a b
#   ... (falta apilar b sobre c)

estados_p2 = [
    [['a', 'b', 'c']],      # Inicial: torre A-B-C
    [['a', 'b'], ['c']],    # unstack c b
    [['a', 'b'], ['c']],    # put-down c (igual, pero brazo libre)
    [['a'], ['c'], ['b']],  # unstack b a
    [['a'], ['c'], ['b']],  # put-down b
    [['c'], ['b'], ['a']],  # ... acciones pick-up a, stack a b, pick-up b, stack b c
    [['c', 'b', 'a']],      # Final: torre C-B-A
]
titulos_p2 = [
    'Inicial\nA-B-C',
    'unstack c b',
    'put-down c',
    'unstack b a',
    'put-down b',
    'pick-up/stack\na, b',
    'Final ✅\nC-B-A',
]
visualizar_blocksworld(estados_p2, titulos_p2, figsize=(16, 3.5))

## 🏗️ Paso 8: Resolver Problema 3 — Torre de 4 bloques

In [ ]:
print("Problema 3: Construir torre D-C-B-A (A en el tope)")
print("  Estado inicial: pilas D-C  y  B-A")
print("  Objetivo: A sobre B, B sobre C, C sobre D")
print()

res_p3_bfs   = planificar(DOMAIN_PDDL, PROBLEM_3, search='bfs')
res_p3_astar = planificar(DOMAIN_PDDL, PROBLEM_3, search='astar', heuristic='hff')

## 📊 Paso 9: Comparativa de planificadores

In [ ]:
# Tabla comparativa
resultados = [
    {'Problema': 'P1 (2 bloques)',  'Algoritmo': 'BFS',       'Pasos': res_p1_bfs['length'],   'Tiempo (ms)': f"{res_p1_bfs['time_ms']:.1f}"},
    {'Problema': 'P2 (3 bloques)',  'Algoritmo': 'BFS',       'Pasos': res_p2_bfs['length'],   'Tiempo (ms)': f"{res_p2_bfs['time_ms']:.1f}"},
    {'Problema': 'P2 (3 bloques)',  'Algoritmo': 'A* + hFF',  'Pasos': res_p2_astar['length'], 'Tiempo (ms)': f"{res_p2_astar['time_ms']:.1f}"},
    {'Problema': 'P3 (4 bloques)',  'Algoritmo': 'BFS',       'Pasos': res_p3_bfs['length'],   'Tiempo (ms)': f"{res_p3_bfs['time_ms']:.1f}"},
    {'Problema': 'P3 (4 bloques)',  'Algoritmo': 'A* + hFF',  'Pasos': res_p3_astar['length'], 'Tiempo (ms)': f"{res_p3_astar['time_ms']:.1f}"},
]

df = pd.DataFrame(resultados)
print("\n=== Comparativa de planificadores ===")
display(df)

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Longitud del plan
for ax, col, ylabel, title in [
    (axes[0], 'Pasos', 'Número de acciones', 'Longitud del plan'),
    (axes[1], 'Tiempo (ms)', 'Tiempo (ms)', 'Tiempo de ejecución'),
]:
    df_plot = df.copy()
    df_plot['Label'] = df_plot['Problema'] + '\n' + df_plot['Algoritmo']
    valores = pd.to_numeric(df_plot[col], errors='coerce')
    colores_bar = ['#3498db' if 'BFS' in alg else '#e74c3c' for alg in df_plot['Algoritmo']]
    bars = ax.bar(range(len(df_plot)), valores, color=colores_bar, alpha=0.85, edgecolor='black')
    ax.set_xticks(range(len(df_plot)))
    ax.set_xticklabels(df_plot['Label'], fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    for bar, val in zip(bars, valores):
        if pd.notna(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    f"{val:.1f}" if isinstance(val, float) else str(int(val)),
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#3498db', label='BFS'),
                       Patch(facecolor='#e74c3c', label='A* + hFF')]
    ax.legend(handles=legend_elements)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Comparativa: BFS vs A*+hFF — Blocksworld', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🌐 Paso 10: Otros dominios PDDL

PDDL es un estándar universal: los mismos planificadores resuelven dominios completamente distintos cambiando solo los archivos de entrada.

| Dominio | Descripción |
|---------|-------------|
| **Blocksworld** | Apilar bloques (este notebook) |
| **Logistics** | Transportar paquetes entre ciudades con camiones y aviones |
| **Rover** | Planificación de misiones de rovers en Marte (NASA) |
| **Gripper** | Robot con pinzas que mueve pelotas entre habitaciones |
| **Satellite** | Secuenciar observaciones de un satélite |

pyperplan incluye muchos de estos dominios en su repositorio. También pueden descargarse de la [IPC (International Planning Competition)](https://ipc.icaps-conference.org/).

In [ ]:
# ============================================================
# Extra: Dominio Gripper — robot mueve pelotas entre cuartos
# ============================================================
GRIPPER_DOMAIN = """
(define (domain gripper-strips)
  (:requirements :strips)
  (:predicates
    (room ?r)
    (ball ?b)
    (gripper ?g)
    (at-robby ?r)
    (at ?b ?r)
    (free ?g)
    (carry ?o ?g)
  )
  (:action move
    :parameters (?from ?to)
    :precondition (and (room ?from) (room ?to) (at-robby ?from))
    :effect (and (at-robby ?to) (not (at-robby ?from)))
  )
  (:action pick
    :parameters (?obj ?room ?gripper)
    :precondition (and (ball ?obj) (room ?room) (gripper ?gripper) (at ?obj ?room) (at-robby ?room) (free ?gripper))
    :effect (and (carry ?obj ?gripper) (not (at ?obj ?room)) (not (free ?gripper)))
  )
  (:action drop
    :parameters (?obj ?room ?gripper)
    :precondition (and (ball ?obj) (room ?room) (gripper ?gripper) (carry ?obj ?gripper) (at-robby ?room))
    :effect (and (at ?obj ?room) (free ?gripper) (not (carry ?obj ?gripper)))
  )
)
"""

GRIPPER_PROBLEM = """
(define (problem gripper-2-balls)
  (:domain gripper-strips)
  (:objects
    rooma roomb
    ball1 ball2
    left right
  )
  (:init
    (room rooma) (room roomb)
    (ball ball1)  (ball ball2)
    (gripper left) (gripper right)
    (at-robby rooma)
    (at ball1 rooma) (at ball2 rooma)
    (free left) (free right)
  )
  (:goal (and (at ball1 roomb) (at ball2 roomb)))
)
"""

print("Dominio Gripper: mover 2 pelotas de RoomaA a RoomB")
res_gripper = planificar(GRIPPER_DOMAIN, GRIPPER_PROBLEM, search='bfs')

## 🎓 Resumen y Conclusiones

### Puntos clave

1. **PDDL** separa claramente el **dominio** (lógica del mundo) del **problema** (instancia concreta), permitiendo reutilizar dominios con distintos problemas.
2. **BFS** garantiza el plan **óptimo en número de acciones**, pero el espacio de búsqueda crece exponencialmente con el número de bloques/acciones.
3. **A* con hFF** (Heurística de Función de Relajación) reduce drásticamente los nodos explorados al estimar el costo restante, sin perder optimalidad.
4. La planificación automatizada es escalable: el mismo planificador resuelve dominios completamente distintos (blocksworld, gripper, rover) sin cambiar el código.
5. PDDL es la base de sistemas reales: planificación de misiones espaciales, robótica, gestión de flujos de trabajo y juegos.

### 🚀 ¿Qué sigue?

- **PDDL 2.1+**: añade fluentes numéricos, duraciones y procesos continuos.
- **Planificación HTN** (Hierarchical Task Networks): descompone tareas en subtareas.
- **Planificación probabilística**: cuando las acciones tienen efectos inciertos (MDPs).
- **Planificación con aprendizaje**: combinar PDDL con aprendizaje automático para estimar mejores heurísticas.

### 📚 Referencias

- Russell & Norvig — *Artificial Intelligence: A Modern Approach* (4th ed.), Cap. 11.
- Helmert, M. (2006). *The Fast Downward Planning System*. JAIR, 26, 191–246.
- pyperplan: https://github.com/aibasel/pyperplan
- IPC (International Planning Competition): https://ipc.icaps-conference.org/
- Ghallab, M. et al. (2004). *Automated Planning: Theory and Practice*. Morgan Kaufmann.

---

*© 2026 Cátedra Inteligencia Artificial — Lic. en Sistemas — FCAD/UNER*  
[![CC BY-SA 4.0](https://licensebuttons.net/l/by-sa/4.0/88x31.png)](https://creativecommons.org/licenses/by-sa/4.0/)